# Stage 5 §V1 — matched-wall value-head-only search on TSP-20 (Colab T4)

**Context.** §V0 PASSED: off-policy rollout distillation took the glimpse head's held-out sibling ranking to rollout parity (depth-1 regret 0.0554 vs rollout 0.0563; Spearman(v,g) 0.081→0.788; depth-2 within 2× of the rollout anchor). V1 now cashes the compute dividend end-to-end: does `leaf_eval='value_head'` with the DISTILLED head, at **equal wall-clock**, match or beat `leaf_eval='rollout'` K=40 — the production leaf eval?

**Arms** (all `val_stage4_mcts.py`, num_test=10000 seed=42, ε=0, τ=const, c_puct=0.05, mcts_batch_size=1000, which=best; per-instance costs saved for paired tests):
- **R** — rollout K=40, distilled ckpt (rollout ignores the head ⇒ doubles as a policy-identity sanity vs the original ckpt). Measures the wall budget W_R.
- **V40** — value_head K=40, distilled head. Measures per-sim wall → K_matched = 40·W_R/W_V40 (rounded to 20, clamped [60, 800]).
- **VM** — value_head K=K_matched, distilled head. **The primary arm.**
- **O40** — value_head K=40, ORIGINAL §H.4 head (the §C.3-style contrast on this policy).

**Pre-registered criteria:**
- **S1 (mechanism):** V40 beats greedy (paired one-sided p<0.01). The original head fails this (§C.3); the distilled head must pass. Report O40 vs greedy as the contrast.
- **S2 (primary, non-inferiority):** paired Δ(VM − R) ≤ +0.002 (≈⅔·SE tie band). Strictly better ⇒ headline; parity ⇒ mechanism validated, prize moves to N≥50 where the multiplier is ~26×.

**Wall estimate:** R ≈ 4 min, V40 ≈ 1 min, VM ≈ 4 min, O40 ≈ 1 min ⇒ **~15 min total**.

**Requires on Drive:** `iter-99_vh_offpolicy.pt` in the §H.4 run dir (produced by the V0 notebook).

## Section 1 — setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/AM_AlphaGoZero'
REPO_DIR = os.path.join(WORKSPACE, 'repo')
RUN_DIR = os.path.join(WORKSPACE, 'outputs', 'tsp_20',
                       'tsp20_k10_mix0p5_step50_100iter_20260522T055640_20260522T055644')
CKPT_ORIG = os.path.join(RUN_DIR, 'iter-99.pt')
CKPT_DIST = os.path.join(RUN_DIR, 'iter-99_vh_offpolicy.pt')
V1_DIR = os.path.join(RUN_DIR, 'v1_matched_wall')
os.makedirs(V1_DIR, exist_ok=True)
for f in (CKPT_ORIG, CKPT_DIST):
    assert os.path.exists(f), f'missing on Drive: {f} — run the V0 notebook first'
print('OK — both checkpoints present;', 'V1_DIR =', V1_DIR)

In [ ]:
REPO_URL = 'https://github.com/LejunZhou/AM_ALPHAGOZERO.git'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} log --oneline -1

%cd {REPO_DIR}
!pip install --quiet pybind11
!pip install --quiet --no-deps -e .
!pip install --quiet numpy scipy tqdm

# --save_costs must be present (Stage 5 §V1 edit).
!grep -q 'save_costs' src/scripts/val_stage4_mcts.py && echo 'OK — save_costs supported'

In [ ]:
import re, subprocess

COMMON = ('--which best --num_test 10000 --seed 42 --eps 0.0 '
          '--temperature_schedule const --c_puct 0.05 --mcts_batch_size 1000')

def run_arm(tag, ckpt, leaf_eval, K):
    npz = os.path.join(V1_DIR, f'{tag}.npz')
    log = os.path.join(V1_DIR, f'{tag}.log')
    cmd = (f'cd {REPO_DIR} && PYTHONPATH=src python src/scripts/val_stage4_mcts.py '
           f'--ckpt {ckpt} --leaf_eval {leaf_eval} --K {K} {COMMON} --save_costs {npz}')
    print(f'=== {tag}: leaf_eval={leaf_eval} K={K} ===')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    open(log, 'w').write(r.stdout + '\n--- STDERR ---\n' + r.stderr)
    tail = r.stdout.strip().splitlines()
    print('\n'.join(tail[-14:]))
    assert r.returncode == 0, f'{tag} FAILED — see {log}'
    m = re.findall(r'wall:\s*([\d.]+)s', r.stdout)
    wall = float(m[-1])
    print(f'[{tag}] MCTS wall = {wall:.1f}s  (npz: {npz})\n')
    return npz, wall

print('runner ready')

## Section 2 — arms R and V40 (sets the wall budget and K_matched)

In [ ]:
NPZ_R, WALL_R = run_arm('armR_rollout_K40', CKPT_DIST, 'rollout', 40)

In [ ]:
import numpy as np
NPZ_V40, WALL_V40 = run_arm('armV40_vh_K40_distilled', CKPT_DIST, 'value_head', 40)

K_MATCHED = int(np.clip(round(40 * WALL_R / WALL_V40 / 20) * 20, 60, 800))
print(f'W_R={WALL_R:.1f}s  W_V40={WALL_V40:.1f}s  ratio={WALL_R/WALL_V40:.2f}x  -> K_MATCHED={K_MATCHED}')

## Section 3 — primary arm VM (matched wall) + contrast O40

In [ ]:
NPZ_VM, WALL_VM = run_arm(f'armVM_vh_K{K_MATCHED}_distilled', CKPT_DIST, 'value_head', K_MATCHED)

In [ ]:
NPZ_O40, WALL_O40 = run_arm('armO40_vh_K40_original', CKPT_ORIG, 'value_head', 40)

## Section 4 — verdict (paired tests on shared val set)

In [ ]:
import numpy as np
from math import erf, sqrt

def load(npz):
    z = np.load(npz)
    return z['greedy_best_model'], z['mcts_best_model']

g_R, R = load(NPZ_R)
g_V, V40 = load(NPZ_V40)
g_M, VM = load(NPZ_VM)
g_O, O40 = load(NPZ_O40)
assert (g_R == g_V).all() and (g_R == g_M).all() and (g_R == g_O).all(), \
    'greedy baselines differ across arms — val sets not identical!'
greedy = g_R

def paired(a, b):
    d = a - b
    n = len(d)
    se = d.std(ddof=1) / sqrt(n)
    t = d.mean() / se
    p_one = 0.5 * (1 + erf(t / sqrt(2)))          # P(diff >= 0) under H0, normal approx (n=10k)
    return d.mean(), se, t, min(p_one, 1 - p_one)

def se(x): return x.std(ddof=1) / sqrt(len(x))

walls = {'greedy': 0.0, 'R': WALL_R, 'V40': WALL_V40, 'VM': WALL_VM, 'O40': WALL_O40}
print(f"{'arm':<38}{'mean':>9}{'SE':>9}{'wall(s)':>9}")
for tag, arr in [('greedy θ★', greedy), ('R rollout K=40', R), ('V40 vh-distilled K=40', V40),
                 (f'VM vh-distilled K={K_MATCHED} (matched)', VM), ('O40 vh-ORIGINAL K=40', O40)]:
    w = walls.get(tag.split()[0].replace('θ★', 'greedy'), 0.0)
    print(f'{tag:<38}{arr.mean():>9.5f}{se(arr):>9.5f}{w:>9.1f}')

print('\npaired diffs (negative = first arm better):')
for name, a, b in [('V40 - greedy', V40, greedy), ('O40 - greedy', O40, greedy),
                   ('V40 - R', V40, R), ('VM - R', VM, R), ('VM - greedy', VM, greedy)]:
    dm, dse, t, p = paired(a, b)
    print(f'  {name:<14} Δ={dm:+.5f}  SE={dse:.5f}  t={t:+.2f}  p_one={p:.2e}')

d1, s1, t1, p1 = paired(V40, greedy)
S1 = (d1 < 0) and (p1 < 0.01)
d2, s2, t2, p2 = paired(VM, R)
S2 = d2 <= 0.002
print(f'\nS1 (mechanism: V40 < greedy, p<0.01):            {"PASS" if S1 else "FAIL"}  (Δ={d1:+.5f}, p={p1:.1e})')
print(f'S2 (primary: VM - R <= +0.002 non-inferiority):  {"PASS" if S2 else "FAIL"}  (Δ={d2:+.5f} @ K_matched={K_MATCHED})')
print('\nVERDICT:', 'V1 PASS — value head validated as amortized rollout at matched wall; proceed to V2 (TSP-50)'
      if (S1 and S2) else 'V1 PARTIAL/FAIL — paste table back for diagnosis (K-curve / c_puct / deeper-tree distribution shift are the suspects)')

## Next steps

Paste the Section 4 table + verdict back to the main thread. Artifacts persist in `RUN_DIR/v1_matched_wall/` (4 npz + 4 logs). On PASS: V2 scaffold — TSP-50 labeling pass on an §E buffer + distill + matched-wall arms against the §E rollout numbers, where the multiplier is ~26× and the parity gap (+0.132 vs Stage-1 greedy) is the target.